# M12.3 — Train-only balanced sampling diagnostics

## A. Hypothesis, scope and effective objective
Before implementing a sampler, compare original G1 train exposure with a **marginal-mixture, with-replacement** schedule. Hypothesis: modest peripheral-pool oversampling may increase exposure of marginally extreme labels while retaining acceptable train-row coverage and avoiding excessive concentration. This diagnostic measures sampling properties, not prediction improvements.

M11, M12.1 and M12.2 remain unchanged. M12.3 would use ordinary per-batch MSE with the M11 architecture, optimizer, preprocessing, label scaler and unweighted validation selection. It must not combine sampling with M12.1/M12.2 weighted losses in the first comparison. All references and policy assessment here use TRAIN labels only; no validation-model selection occurs.

If row i has draw probability π_i, expected batch loss is Σ_i π_i ℓ_i = (1/N)Σ_i (Nπ_i)ℓ_i, where ℓ_i is the mean squared **standardized** error across the three targets. Thus replacement sampling changes the effective risk. It is **not** an unweighted empirical-risk estimator over the original uniform train distribution. No inverse-probability correction is proposed. Even matching expected risks does not equate stochastic gradients, example diversity, optimizer trajectories or generalization.

Reuse [M12.2 train diagnostics](M12_2_marginal_weighting_diagnostics.ipynb), frozen M12.0 geometry and existing weighting functions. This notebook is unexecuted, CPU-only, writes nothing, and contains no actual training sampler implementation. Simulations below generate row-ID schedules in memory only when manually invoked.


## B. Frozen train labels, identities and reference reproduction
Use M12.0's src.paths/data-root convention: CBC_PE_DATA_ROOT overrides the frozen M11 config. Read only train indices, split membership metadata, train-only scaler metadata and physical HDF5 TRAIN labels in bounded sorted reads. Never read X or any nontrain target. Hash the small split/scaler files against M12.0; maintain the original row-ID mapping.

The existing marginal implementation reproduces the original M12.2 distances |y_k−μ_train,k|/σ_train,k, population σ (ddof=0), right-side empirical CDF, including ties and self. Existing Mahalanobis implementation supplies the unchanged global reference and M12.1 weights. No new scaler or alternative statistical fit is introduced. Labels stay [chirp_mass, total_mass, chi_eff].


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
import h5py

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from cbc_pe")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import resolve_data_root, resolve_processed_artifact
from src.models.sample_weighting import mahalanobis_sample_weights, marginal_label_weights, array_hash

LABELS = ["chirp_mass", "total_mass", "chi_eff"]
CONFIG_PATH = ROOT / "configs/experiments/train_m11_final_G1_500k.json"
config = json.loads(CONFIG_PATH.read_text())
DATA_ROOT = resolve_data_root(config_data_root=config["data_root"])
DATASET_ID = "M11_final_G1_500k"
PATHS = {k:resolve_processed_artifact(data_root=DATA_ROOT,dataset_id=DATASET_ID,
    file_name=config["dataset"][k],allow_legacy_flat=False)
    for k in ("dataset_file","split_file","label_stats_file")}
CONTRACT_PATH = ROOT / "notebooks/m12/outputs/m12_0_baseline_contract.json"
SEED = 123
EPOCH_DRAWS = 400000
CUMULATIVE_EPOCHS = [1,2,5,10,25]
MIN_REGION_ROWS = 100
# Prospective diagnostic budgets, not scientific validation thresholds.
MIN_WEIGHT_ESS_FRACTION = 0.90
MIN_COVERAGE_RELATIVE_TO_UNIFORM_WR = 0.95
MAX_EFFECTIVE_WEIGHT = 2.0

def require(ok,message):
    if not ok:
        raise ValueError(message)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

require(config["dataset"]["dataset_id"] == DATASET_ID, "Wrong dataset config")
require(all(p.is_file() for p in PATHS.values()) and CONTRACT_PATH.is_file(), "Missing frozen inputs")
frozen = json.loads(CONTRACT_PATH.read_text())
require(frozen["dataset_id"] == DATASET_ID and frozen["labels"] == LABELS
        and frozen["schema_version"] == 1, "Wrong M12.0 contract")
geometry = frozen["geometry"]
for key in ("split_file","label_stats_file"):
    require(file_hash(PATHS[key]) == frozen["provenance"]["local_artifacts"][key]["sha256"],
            f"Frozen artifact changed: {key}")
del frozen  # No historical evaluation metrics are used.
with np.load(PATHS["split_file"],allow_pickle=False) as z:
    train_ids = z["train_idx"].copy()
    require(int(z["seed"].item()) == 123, "Split seed changed")
require(train_ids.shape == (400000,) and np.issubdtype(train_ids.dtype,np.integer)
        and len(np.unique(train_ids)) == len(train_ids), "Invalid or duplicate train IDs")
train_ids = np.sort(train_ids.astype(np.int64))
require(((train_ids >= 0) & (train_ids < 500000)).all(), "Train IDs out of range")
require(array_hash(train_ids) == geometry["train_indices_sha256"], "Changed train membership")
with np.load(PATHS["label_stats_file"],allow_pickle=False) as z:
    require(z["label_names"].tolist() == LABELS, "Scaler labels")
    require(np.array_equal(np.sort(z["train_idx"]),train_ids), "Scaler train IDs")
    saved_mean, saved_std = z["y_mean"].copy(), z["y_std"].copy()
require(saved_mean.shape == saved_std.shape == (3,) and np.isfinite(saved_mean).all()
        and np.isfinite(saved_std).all() and (saved_std > 0).all(), "Invalid saved scaler")
with h5py.File(PATHS["dataset_file"],"r") as h:
    require(h.attrs.get("experiment_id") == DATASET_ID
            and h.attrs.get("labels") == ",".join(LABELS), "Wrong HDF5 contract")
    require(h["y"].shape == (500000,3), "Wrong label shape")
    membership = h["split"].asstr()[:]
    require(np.array_equal(np.flatnonzero(membership == "train"),train_ids), "HDF5 train mismatch")
    train_y = np.empty((len(train_ids),3),dtype=np.float64)
    for start in range(0,len(train_ids),4096):
        train_y[start:start+4096] = h["y"][train_ids[start:start+4096]]
del membership
require(np.isfinite(train_y).all(), "Nonfinite train labels")
N = len(train_ids)
BATCH_SIZE = int(config["training"]["batch_size"])
DROP_LAST = bool(config["training"]["drop_last"])
require(BATCH_SIZE == 256 and DROP_LAST and config["training"]["data_loading_mode"] == "hdf5_batch_slices",
        "Frozen M11 exposure/loading assumptions changed")
require(config["training"]["shuffle_batches"] and not config["training"]["shuffle_within_batch"],
        "M11 shuffle assumptions changed")
FULL_BATCHES = EPOCH_DRAWS // BATCH_SIZE
CONSUMED_DRAWS = FULL_BATCHES * BATCH_SIZE
DROPPED_DRAWS = EPOCH_DRAWS - CONSUMED_DRAWS
provenance = dict(dataset_id=DATASET_ID,train_ids_sha256=array_hash(train_ids),
    train_labels_sha256=array_hash(train_y),contract_sha256=file_hash(CONTRACT_PATH),
    config_sha256=file_hash(CONFIG_PATH),split_sha256=file_hash(PATHS["split_file"]),
    scaler_sha256=file_hash(PATHS["label_stats_file"]),
    weighting_source_sha256=file_hash(ROOT/"src/models/sample_weighting.py"),
    loader_source_sha256=file_hash(ROOT/"src/models/hdf5_batch_dataset.py"),
    seed=SEED,nominal_draws=EPOCH_DRAWS,consumed_draws=CONSUMED_DRAWS,
    batch_size=BATCH_SIZE,drop_last=DROP_LAST,
    versions=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,h5py=h5py.__version__))
global_w, global_q, global_meta = mahalanobis_sample_weights(train_y,alpha=.5)
marginal_w, marginal_q, marginal_meta = marginal_label_weights(train_y,alpha=.5)
for key in ("n_train","rank","pinv_rcond","reference_distance_sha256"):
    require(global_meta[key] == geometry[key], f"Global reference mismatch: {key}")
for key in ("mean","std_ddof0","covariance_ddof1","scaled_covariance","precision"):
    np.testing.assert_allclose(global_meta[key],geometry[key],rtol=1e-12,atol=1e-12)
for key in ("mean","std_ddof0"):
    np.testing.assert_allclose(marginal_meta[key],geometry[key],rtol=1e-12,atol=1e-12)
require(marginal_q.shape == (N,3) and np.isfinite(marginal_q).all()
        and ((marginal_q >= 0) & (marginal_q <= 1)).all(), "Invalid marginal CDF")
provenance["global_reference"] = global_meta
provenance["marginal_reference"] = marginal_meta
del train_y
print(dict(train_rows=N,nominal_draws=EPOCH_DRAWS,consumed_draws=CONSUMED_DRAWS,
           optimizer_steps=FULL_BATCHES,discarded_draws=DROPPED_DRAWS))


{'train_rows': 400000, 'nominal_draws': 400000, 'consumed_draws': 399872, 'optimizer_steps': 1562, 'discarded_draws': 128}


## C. Pools, mixture probability and original-exposure controls
For each label k, peripheral P_k means q_k≥0.8 and **central C_k is its complement q_k<0.8**. This task's central definition differs deliberately from the older P50 diagnostic. Ties can make peripheral fractions differ from exactly 20%; report actual pool counts.

Choose k uniformly from three labels, choose P_k with probability p or C_k with probability 1−p, then choose a row uniformly inside that pool. Pools overlap across labels; probabilities add:
    
π_i(p) = (1/3) Σ_k [p·1(i∈P_k)/n_Pk + (1−p)·1(i∈C_k)/n_Ck].

All six pools must be nonempty. Effective scalar weights are a_i=Nπ_i, not the probability of being unique in an epoch. Compare p=0.35 and p=0.50, plus p=0.20 as a sanity control. A common p=.20 is exactly uniform only if all actual peripheral fractions equal .20. A mathematical check choosing p_k=n_Pk/N for each label must produce exact uniform probabilities; this is a check, not a fourth candidate.

Controls distinguish (i) ideal original uniform exposure, each row once, (ii) actual M11 block exposure after drop_last (same final 128 sorted train IDs omitted each epoch), and (iii) uniform independent draws **with replacement**. The third isolates replacement-induced duplicate/coverage costs from balancing. Do not misdescribe original M11 sampling as with-replacement.


In [2]:
peripheral = marginal_q >= .8
pool_positions = {(j,side):np.flatnonzero(peripheral[:,j] == side)
                  for j in range(3) for side in (False,True)}
require(all(len(v) > 0 for v in pool_positions.values()), "Empty central/peripheral pool")
pool_counts = pd.DataFrame([dict(label=label,central=len(pool_positions[j,False]),
    peripheral=len(pool_positions[j,True]),peripheral_fraction=peripheral[:,j].mean())
    for j,label in enumerate(LABELS)])
display(pool_counts)

def mixture_probabilities(p):
    per_label_p = np.broadcast_to(np.asarray(p,dtype=np.float64),(3,))
    require(np.isfinite(per_label_p).all() and ((per_label_p > 0) & (per_label_p < 1)).all(),
            "Selection probabilities must be strictly inside (0,1)")
    probability = np.zeros(N,dtype=np.float64)
    for j in range(3):
        for side in (False,True):
            pool = pool_positions[j,side]
            selection = per_label_p[j] if side else 1-per_label_p[j]
            probability[pool] += selection/(3*len(pool))
    require(np.isfinite(probability).all() and (probability > 0).all(), "Invalid row probabilities")
    np.testing.assert_allclose(probability.sum(),1.,rtol=1e-12,atol=1e-12)
    return probability

np.testing.assert_allclose(mixture_probabilities(peripheral.mean(axis=0)),1/N,rtol=1e-12,atol=1e-15)
POLICIES = {"uniform_WR":None,"mixture_p020":.20,"mixture_p035":.35,"mixture_p050":.50}
POLICY_CODES = {"uniform_WR":0,"mixture_p020":20,"mixture_p035":35,"mixture_p050":50}
probabilities = {name:np.full(N,1/N) if p is None else mixture_probabilities(p)
                 for name,p in POLICIES.items()}
m11_consumed_mask = np.arange(N) < CONSUMED_DRAWS
regions = {label+"_P80":peripheral[:,j] for j,label in enumerate(LABELS)}
regions["global_Mahalanobis_P80"] = global_q >= .8
for j,label in enumerate(LABELS[:2]):
    regions[label+"_peripheral_spin_central"] = peripheral[:,j] & ~peripheral[:,2]
    regions[label+"_central_spin_peripheral"] = ~peripheral[:,j] & peripheral[:,2]
# No Cartesian bins: only named marginal/global and four discordant masks.
display(pd.DataFrame([dict(region=name,n=int(mask.sum()),fraction=mask.mean(),
    sufficient_count=int(mask.sum()) >= MIN_REGION_ROWS) for name,mask in regions.items()]))
print("p=.20 departure from uniform (ties/count effects):",
      float(np.max(np.abs(N*probabilities["mixture_p020"]-1))))


,label,central,peripheral,peripheral_fraction
0,chirp_mass,319999,80001,0.200002
1,total_mass,319999,80001,0.200002
2,chi_eff,319999,80001,0.200002


,region,n,fraction,sufficient_count
0,chirp_mass_P80,80001,0.200002,True
1,total_mass_P80,80001,0.200002,True
2,chi_eff_P80,80001,0.200002,True
3,global_Mahalanobis_P80,80001,0.200002,True
4,chirp_mass_peripheral_spin_central,64739,0.161848,True
5,chirp_mass_central_spin_peripheral,64739,0.161848,True
6,total_mass_peripheral_spin_central,65693,0.164233,True
7,total_mass_central_spin_peripheral,65693,0.164233,True


p=.20 departure from uniform (ties/count effects): 1.2499843751845319e-05


## D. Effective scalar weights and relation to M12.1/M12.2
Compare exact a=Nπ with M12.1's global Mahalanobis weights using aligned Pearson correlation, mean absolute differences and quantiles. Weight-based ESS = (Σa)²/Σa² = 1/Σπ², measured in rows, is a probability-concentration summary; it is not realized unique coverage or a training-uncertainty estimate.

Mixture weights depend on overlapping binary peripheral memberships, with jumps at P80. M12.1 smoothly weights the global joint-distance rank. Both are scalar per-row emphasis on all outputs. M12.2 instead has a different coefficient for each row/target; sampling a row increases exposure of **all** its targets. A single scalar probability generally cannot reproduce its three independent column weights. Report column-wise differences without claiming equivalence. Exact weighting-risk equivalence is an expectation for fixed predictions, not an equivalence of minibatches or AdamW trajectories.


In [3]:
weight_rows = []
assignment_rows = []
for name,pi in probabilities.items():
    effective = N*pi
    np.testing.assert_allclose(effective.mean(),1.,rtol=1e-12)
    q = np.quantile(effective,[0,.1,.5,.9,.99,1])
    weight_rows.append(dict(policy=name,mean=effective.mean(),minimum=q[0],q10=q[1],
        median=q[2],q90=q[3],q99=q[4],maximum=q[5],weight_ESS=1/np.dot(pi,pi),
        weight_ESS_fraction=1/(N*np.dot(pi,pi))))
    for reference,weights in [("M12.1 global",global_w),
                              *[(f"M12.2 {label}",marginal_w[:,j]) for j,label in enumerate(LABELS)]]:
        corr = (np.corrcoef(effective,weights)[0,1]
                if np.ptp(effective) > 1e-12 and np.ptp(weights) > 1e-12 else np.nan)
        assignment_rows.append(dict(policy=name,reference=reference,pearson=corr,
            mean_abs_difference=np.mean(np.abs(effective-weights)),
            q90_abs_difference=np.quantile(np.abs(effective-weights),.9)))
weight_summary = pd.DataFrame(weight_rows).set_index("policy")
display(weight_summary)
display(pd.DataFrame(assignment_rows))
# Undefined correlation with constant uniform weights is retained as NaN.
del effective


,mean,minimum,q10,median,q90,q99,maximum,weight_ESS,weight_ESS_fraction
policy,,,,,,,,,
uniform_WR,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,400000.000000,1.000000
mixture_p020,1.0,0.999988,0.999993,1.000003,1.000003,1.000003,1.000003,399999.999992,1.000000
mixture_p035,1.0,0.812503,0.812503,0.812503,1.437486,1.749978,1.749978,373699.579622,0.934249
mixture_p050,1.0,0.625002,0.625002,0.625002,1.874980,2.499969,2.499969,312129.653214,0.780324


,policy,reference,pearson,mean_abs_difference,q90_abs_difference
0,uniform_WR,M12.1 global,NaN,0.100000,0.180000
1,uniform_WR,M12.2 chirp_mass,NaN,0.100000,0.180000
2,uniform_WR,M12.2 total_mass,NaN,0.100000,0.180000
3,uniform_WR,M12.2 chi_eff,NaN,0.100000,0.180000
4,mixture_p020,M12.1 global,-0.641846,0.100003,0.180004
5,mixture_p020,M12.2 chirp_mass,-0.643403,0.100002,0.180005
6,mixture_p020,M12.2 total_mass,-0.617339,0.100002,0.180005
7,mixture_p020,M12.2 chi_eff,-0.300992,0.100001,0.180002
8,mixture_p035,M12.1 global,0.641846,0.157225,0.364578
9,mixture_p035,M12.2 chirp_mass,0.643403,0.171730,0.293769


## E. Expected coverage and multiplicities
For m independent replacement draws with fixed π:
- C_i is row i's multiplicity, marginally Binomial(m,π_i).
- Expected distinct rows U=Σ_i[1−(1−π_i)^m]; expected omitted rows N−U.
- Duplicate draws beyond the first occurrence = m−U, not the number of distinct repeated rows.
- Expected distinct repeated rows = Σ_i[1−Pr(C_i=0)−Pr(C_i=1)].
- Realized empirical-count ESS = m²/Σ_i C_i², reported separately; its expectation is **not** obtained by substituting E[C_i²] into this ratio.
- Multiplicity histograms include all original train rows, including zero-count omissions. Exact expected categories are 0,1,2,3,4,5–9,10+.

Use stable log1p/expm1 calculations. Compare all 400000 generated draws and the first 399872 consumed draws, yielding 1562 optimizer steps under batch size 256/drop_last. The last 128 random schedule positions would be discarded, not the last 128 sorted IDs. No actual optimizer is run.


In [4]:
def expected_occupancy(pi,m):
    p0 = np.exp(m*np.log1p(-pi))
    p1 = p0*m*pi/(1-pi)
    unique = float(np.sum(-np.expm1(m*np.log1p(-pi))))
    return dict(expected_unique=unique,expected_unique_fraction=unique/N,
        expected_omissions=N-unique,expected_duplicate_draws=m-unique,
        expected_repeated_rows=float(np.sum(1-p0-p1)))

def expected_histogram(pi,m):
    pk = np.exp(m*np.log1p(-pi))
    counts = []
    for k in range(10):
        counts.append(float(pk.sum()))
        pk = pk*((m-k)/(k+1))*pi/(1-pi)
    values = counts[:5]+[sum(counts[5:10]),max(0.,N-sum(counts))]
    return dict(zip(["0","1","2","3","4","5-9","10+"],values))

expected_rows = []
for name,pi in probabilities.items():
    for scope,m in (("nominal",EPOCH_DRAWS),("consumed",CONSUMED_DRAWS)):
        expected_rows.append(dict(policy=name,scope=scope,draws=m,**expected_occupancy(pi,m)))
expected_coverage = pd.DataFrame(expected_rows)
display(expected_coverage)
display(pd.DataFrame([
    dict(policy="original_uniform_once",draws=N,unique=N,omissions=0,duplicate_draws=0,
         note="ideal exposure before drop_last"),
    dict(policy="M11_actual_blocks",draws=CONSUMED_DRAWS,unique=CONSUMED_DRAWS,
         omissions=N-CONSUMED_DRAWS,duplicate_draws=0,note="1562 steps; same sorted suffix omitted")
]))


,policy,scope,draws,expected_unique,expected_unique_fraction,expected_omissions,expected_duplicate_draws,expected_repeated_rows
0,uniform_WR,nominal,400000,252848.407471,0.632121,147151.592529,147151.592529,105696.447063
1,uniform_WR,consumed,399872,252801.311368,0.632003,147198.688632,147070.688632,105649.358436
2,mixture_p020,nominal,400000,252848.407470,0.632121,147151.592530,147151.592530,105696.447063
3,mixture_p020,consumed,399872,252801.311366,0.632003,147198.688634,147070.688634,105649.358436
4,mixture_p035,nominal,400000,248113.557045,0.620284,151886.442955,151886.442955,105330.320264
5,mixture_p035,consumed,399872,248067.859204,0.620170,151932.140796,151804.140796,105284.936552
6,mixture_p050,nominal,400000,235123.922844,0.587810,164876.077156,164876.077156,103717.252500
7,mixture_p050,consumed,399872,235081.866252,0.587705,164918.133748,164790.133748,103676.578017


,policy,draws,unique,omissions,duplicate_draws,note
0,original_uniform_once,400000,400000,0,0,ideal exposure before drop_last
1,M11_actual_blocks,399872,399872,128,0,1562 steps; same sorted suffix omitted


## F. Deterministic one-epoch schedule simulation and induced exposure
A notebook-local function implements only the specified **diagnostic** three-stage draws: choose label (1/3), peripheral/central, then a uniform pool position, all with replacement. It is not connected to a DataLoader or training loop. Independent policy/epoch streams use SeedSequence([seed, policy_code, epoch]); changing epoch must change the stream. Short deterministic replays verify reproducibility. Full schedules are hashed but not persisted, and only one is retained temporarily.

Mapping is positions → original train_ids. The source universe must have unique IDs; the schedule intentionally contains duplicates. Every drawn ID must map back exactly to its original training row. We establish unique **dataset row** identity, not uniqueness of underlying astrophysical sources.

Expected event frequency is Σπ_i·1(region_i). Simulated frequency uses all draws or only consumed draws. Label-selection probability p is not generally the observed peripheral frequency for each label because pools overlap and mass/spin coordinates correlate. Compare each event against original full-train and actual M11 consumed exposure, not just p. Sparse-region frequencies are exact descriptive counts, but must not drive decisions alone.


In [5]:
def simulate_positions(policy,count,epoch=0):
    rng = np.random.default_rng(np.random.SeedSequence([SEED,POLICY_CODES[policy],epoch]))
    p = POLICIES[policy]
    if p is None:
        return rng.integers(0,N,size=count,dtype=np.int64)
    selected_label = rng.integers(0,3,size=count)
    selected_side = rng.random(count) < p
    positions = np.empty(count,dtype=np.int64)
    for j in range(3):
        for side in (False,True):
            where = np.flatnonzero((selected_label == j) & (selected_side == side))
            pool = pool_positions[j,side]
            positions[where] = pool[rng.integers(0,len(pool),size=len(where))]
    return positions

def realized_occupancy(counts):
    m = int(counts.sum()); unique = int(np.count_nonzero(counts))
    return dict(realized_unique=unique,realized_unique_fraction=unique/N,realized_omissions=N-unique,
        realized_duplicate_draws=m-unique,realized_repeated_rows=int(np.sum(counts >= 2)),
        empirical_count_ESS=float(m*m/np.dot(counts.astype(float),counts)),
        max_multiplicity=int(counts.max()),multiplicity_q50=float(np.quantile(counts,.5)),
        multiplicity_q90=float(np.quantile(counts,.9)),multiplicity_q99=float(np.quantile(counts,.99)))

def realized_histogram(counts):
    return {**{str(k):int(np.sum(counts == k)) for k in range(5)},
            "5-9":int(np.sum((counts >= 5) & (counts <= 9))),"10+":int(np.sum(counts >= 10))}

coverage_rows, multiplicity_rows, exposure_rows = [], [], []
schedule_hashes = {}
for name,pi in probabilities.items():
    replay = simulate_positions(name,2048)
    require(np.array_equal(replay,simulate_positions(name,2048)), "RNG replay mismatch")
    require(not np.array_equal(replay,simulate_positions(name,2048,epoch=1)), "Repeated epoch stream")
    positions = simulate_positions(name,EPOCH_DRAWS)
    require(positions.shape == (EPOCH_DRAWS,) and ((positions >= 0) & (positions < N)).all(),
            "Invalid schedule positions")
    row_ids = train_ids[positions]
    require(np.array_equal(np.searchsorted(train_ids,row_ids),positions), "Row-ID alignment failed")
    schedule_hashes[name] = array_hash(row_ids)
    for scope,m in (("nominal",EPOCH_DRAWS),("consumed",CONSUMED_DRAWS)):
        counts = np.bincount(positions[:m],minlength=N)
        require(counts.sum() == m, "Schedule draw count mismatch")
        coverage_rows.append(dict(policy=name,scope=scope,draws=m,**realized_occupancy(counts)))
        observed_hist = realized_histogram(counts)
        analytic_hist = expected_histogram(pi,m)
        require(sum(observed_hist.values()) == N, "Multiplicity histogram incomplete")
        np.testing.assert_allclose(sum(analytic_hist.values()),N,rtol=1e-12,atol=1e-6)
        for category in observed_hist:
            multiplicity_rows.append(dict(policy=name,scope=scope,multiplicity=category,
                expected_rows=analytic_hist[category],simulated_rows=observed_hist[category]))
        for region,mask in regions.items():
            uniform = float(mask.mean())
            expected = float(pi[mask].sum())
            exposure_rows.append(dict(policy=name,scope=scope,region=region,original_rows=int(mask.sum()),
                original_uniform_frequency=uniform,M11_consumed_frequency=float(mask[m11_consumed_mask].mean()),
                expected_frequency=expected,simulated_frequency=float(counts[mask].sum()/m),
                expected_lift=expected/uniform if uniform > 0 else np.nan,
                expected_iid_frequency_sd=float(np.sqrt(expected*(1-expected)/m))))
    del positions,row_ids,counts
coverage = expected_coverage.merge(pd.DataFrame(coverage_rows),on=["policy","scope","draws"],validate="one_to_one")
exposure = pd.DataFrame(exposure_rows)
multiplicity = pd.DataFrame(multiplicity_rows)
display(coverage)
display(multiplicity)
display(exposure[exposure.scope.eq("consumed")])
provenance["schedule_sha256"] = schedule_hashes
# SD is conditional multinomial sampling variation, not model-performance uncertainty.


,policy,scope,draws,expected_unique,expected_unique_fraction,expected_omissions,expected_duplicate_draws,expected_repeated_rows,realized_unique,realized_unique_fraction,realized_omissions,realized_duplicate_draws,realized_repeated_rows,empirical_count_ESS,max_multiplicity,multiplicity_q50,multiplicity_q90,multiplicity_q99
0,uniform_WR,nominal,400000,252848.407471,0.632121,147151.592529,147151.592529,105696.447063,253156,0.632890,146844,146844,105686,200450.010273,8,1.0,2.0,4.0
1,uniform_WR,consumed,399872,252801.311368,0.632003,147198.688632,147070.688632,105649.358436,253102,0.632755,146898,146770,105639,200409.116913,8,1.0,2.0,4.0
2,mixture_p020,nominal,400000,252848.407470,0.632121,147151.592530,147151.592530,105696.447063,252294,0.630735,147706,147706,105915,199419.191604,8,1.0,2.0,4.0
3,mixture_p020,consumed,399872,252801.311366,0.632003,147198.688634,147070.688634,105649.358436,252240,0.630600,147760,147632,105868,199380.547431,8,1.0,2.0,4.0
4,mixture_p035,nominal,400000,248113.557045,0.620284,151886.442955,151886.442955,105330.320264,248066,0.620165,151934,151934,105421,193255.853841,10,1.0,2.0,4.0
5,mixture_p035,consumed,399872,248067.859204,0.620170,151932.140796,151804.140796,105284.936552,248017,0.620043,151983,151855,105373,193219.940963,10,1.0,2.0,4.0
6,mixture_p050,nominal,400000,235123.922844,0.587810,164876.077156,164876.077156,103717.252500,234842,0.587105,165158,165158,103673,175023.299977,10,1.0,3.0,5.0
7,mixture_p050,consumed,399872,235081.866252,0.587705,164918.133748,164790.133748,103676.578017,234800,0.587000,165200,165072,103636,174999.361264,10,1.0,3.0,5.0


,policy,scope,multiplicity,expected_rows,simulated_rows
0,uniform_WR,nominal,0,147151.592529,146844
1,uniform_WR,nominal,1,147151.960409,147470
2,uniform_WR,nominal,2,73575.980204,73724
3,uniform_WR,nominal,3,24525.265421,24420
4,uniform_WR,nominal,4,6131.285699,6109
5,uniform_WR,nominal,5-9,1463.871172,1433
6,uniform_WR,nominal,10+,0.044566,0
7,uniform_WR,consumed,0,147198.688632,146898
8,uniform_WR,consumed,1,147151.952932,147463
9,uniform_WR,consumed,2,73552.432094,73698


,policy,scope,region,original_rows,original_uniform_frequency,M11_consumed_frequency,expected_frequency,simulated_frequency,expected_lift,expected_iid_frequency_sd
8,uniform_WR,consumed,chirp_mass_P80,80001,0.200002,0.200004,0.200002,0.199434,1.000000,0.000633
9,uniform_WR,consumed,total_mass_P80,80001,0.200002,0.199999,0.200002,0.199556,1.000000,0.000633
10,uniform_WR,consumed,chi_eff_P80,80001,0.200002,0.200009,0.200002,0.199784,1.000000,0.000633
11,uniform_WR,consumed,global_Mahalanobis_P80,80001,0.200002,0.200009,0.200002,0.199809,1.000000,0.000633
12,uniform_WR,consumed,chirp_mass_peripheral_spin_central,64739,0.161848,0.161847,0.161848,0.161407,1.000000,0.000582
13,uniform_WR,consumed,chirp_mass_central_spin_peripheral,64739,0.161848,0.161852,0.161848,0.161757,1.000000,0.000582
14,uniform_WR,consumed,total_mass_peripheral_spin_central,65693,0.164233,0.164233,0.164233,0.163765,1.000000,0.000586
15,uniform_WR,consumed,total_mass_central_spin_peripheral,65693,0.164233,0.164243,0.164233,0.163992,1.000000,0.000586
24,mixture_p020,consumed,chirp_mass_P80,80001,0.200002,0.200004,0.200001,0.199354,0.999993,0.000633
25,mixture_p020,consumed,total_mass_P80,80001,0.200002,0.199999,0.200001,0.199564,0.999993,0.000633


## G. Cumulative coverage without epoch×row matrices
Assume **independent newly generated schedules each epoch** and unchanged row probabilities. After E epochs, replace m by E×400000 for nominal draws or E×399872 for consumed draws in the analytic occupancy formula. The results are expectations, not a simulation or proof of independence. Replaying one identical epoch schedule would never increase its unique coverage.

The actual historical M11 sorted-block loader repeatedly omits the same suffix under drop_last, so its cross-epoch unique coverage stays at 399872; shuffling batch order alone does not change membership. A hypothetical uniform fresh permutation each epoch would differ, but is not the historical loader and is not substituted here. Replacement can eventually cover almost all rows while still seeing fewer distinct examples in each epoch.


In [6]:
cumulative_rows = []
for name,pi in probabilities.items():
    for epochs in CUMULATIVE_EPOCHS:
        for scope,draws_per_epoch in (("nominal",EPOCH_DRAWS),("consumed",CONSUMED_DRAWS)):
            cumulative_rows.append(dict(policy=name,epochs=epochs,scope=scope,
                optimizer_steps=epochs*FULL_BATCHES if scope == "consumed" else np.nan,
                **expected_occupancy(pi,epochs*draws_per_epoch)))
cumulative = pd.DataFrame(cumulative_rows)
display(cumulative[["policy","epochs","scope","optimizer_steps","expected_unique",
                    "expected_unique_fraction","expected_omissions"]])
print("Historical M11 consumed unique rows across these epochs:",CONSUMED_DRAWS)
# Tables only: no extra figures, multi-epoch schedules or output files.


,policy,epochs,scope,optimizer_steps,expected_unique,expected_unique_fraction,expected_omissions
0,uniform_WR,1,nominal,NaN,252848.407471,0.632121,147151.592529
1,uniform_WR,1,consumed,1562.0,252801.311368,0.632003,147198.688632
2,uniform_WR,2,nominal,NaN,345866.022041,0.864665,54133.977959
3,uniform_WR,2,consumed,3124.0,345831.365162,0.864578,54168.634838
4,uniform_WR,5,nominal,NaN,397304.838045,0.993262,2695.161955
5,uniform_WR,5,consumed,7810.0,397300.522329,0.993251,2699.477671
6,uniform_WR,10,nominal,NaN,399981.840255,0.999955,18.159745
7,uniform_WR,10,consumed,15620.0,399981.782051,0.999954,18.217949
8,uniform_WR,25,nominal,NaN,399999.999994,1.000000,0.000006
9,uniform_WR,25,consumed,39050.0,399999.999994,1.000000,0.000006


Historical M11 consumed unique rows across these epochs: 399872


## H. Prospective train-side decision rule
Use exact probabilities/analytic coverage, not a favorable simulated seed. Prefer the smallest departure from uniform exposure among satisfactory candidates: p=.35 before p=.50. The .20 mixture and uniform_WR are controls, not candidates.

Declare the following transparent **diagnostic design budgets before examining results**, not validated performance guarantees:
1. Every marginal P80 event gains expected exposure over original uniform train exposure.
2. Weight ESS/N ≥0.90, limiting concentration.
3. Expected consumed unique coverage is ≥95% of uniform-with-replacement coverage at the same draw count, limiting the **additional** loss from balancing.
4. Max Nπ_i ≤2, limiting individual-row emphasis.
5. All four discordant regions have at least 100 original train rows for interpretable contrasts.

Replacement's substantial coverage loss versus original without-replacement exposure remains a separate cost even if these budgets pass. Review those absolute coverage/duplicate counts rather than treating the comparison to uniform_WR as sufficient justification. The budgets express a conservative first-trial preference; changing them after inspecting outputs would constitute a new documented diagnostic decision, not a retroactive pass.

The cell below selects one provisional candidate or explicitly reports neither satisfactory when manually run. Until then selection is **pending**, with no fabricated train-side result. Passing proposes a sampling policy for later review only; it does not authorize a GPU run or claim improved validation predictions.


In [7]:
uniform_expected = expected_occupancy(probabilities["uniform_WR"],CONSUMED_DRAWS)["expected_unique"]
decision_rows = []
discordant_names = [name for name in regions if "_spin_" in name]
require(len(discordant_names) == 4,"Expected four separate mass/spin discordant regions")
for name in ("mixture_p035","mixture_p050"):
    pi = probabilities[name]
    gains = [float(pi[peripheral[:,j]].sum()-peripheral[:,j].mean()) for j in range(3)]
    coverage_ratio = expected_occupancy(pi,CONSUMED_DRAWS)["expected_unique"]/uniform_expected
    ess_fraction = 1/(N*np.dot(pi,pi))
    max_weight = float((N*pi).max())
    checks = dict(all_marginals_gain=all(g > 0 for g in gains),
        concentration_ok=ess_fraction >= MIN_WEIGHT_ESS_FRACTION,
        coverage_ok=coverage_ratio >= MIN_COVERAGE_RELATIVE_TO_UNIFORM_WR,
        max_weight_ok=max_weight <= MAX_EFFECTIVE_WEIGHT,
        discordant_support_ok=all(regions[r].sum() >= MIN_REGION_ROWS for r in discordant_names))
    decision_rows.append(dict(policy=name,p=POLICIES[name],ess_fraction=ess_fraction,
        consumed_coverage_vs_uniform_WR=coverage_ratio,max_effective_weight=max_weight,
        **{label+"_peripheral_gain":g for label,g in zip(LABELS,gains)},
        **checks,eligible=all(checks.values())))
decision_table = pd.DataFrame(decision_rows)
display(decision_table)
eligible = decision_table.loc[decision_table.eligible].sort_values("p")
selected_policy = eligible.iloc[0].policy if len(eligible) else None
if selected_policy is None:
    print("Neither candidate satisfies the declared train-side diagnostic budgets. No sampler/training proposed.")
else:
    print("Provisional train-side policy:",selected_policy,
          "— smallest peripheral-selection probability meeting the declared budgets; not a GPU authorization.")
decision_record = dict(selected_policy=selected_policy,stage="train-only diagnostic",
    criteria=dict(min_weight_ESS_fraction=MIN_WEIGHT_ESS_FRACTION,
        min_coverage_relative_uniform_WR=MIN_COVERAGE_RELATIVE_TO_UNIFORM_WR,
        max_effective_weight=MAX_EFFECTIVE_WEIGHT,min_region_rows=MIN_REGION_ROWS),
    probabilities_sha256={name:array_hash(pi) for name,pi in probabilities.items()},
    decisions=decision_table.to_dict("records"),scientific_improvement=None)


,policy,p,ess_fraction,consumed_coverage_vs_uniform_WR,max_effective_weight,chirp_mass_peripheral_gain,total_mass_peripheral_gain,chi_eff_peripheral_gain,all_marginals_gain,concentration_ok,coverage_ok,max_weight_ok,discordant_support_ok,eligible
0,mixture_p035,0.35,0.934249,0.981276,1.749978,0.088931,0.088186,0.048100,True,True,True,True,True,True
1,mixture_p050,0.50,0.780324,0.929908,2.499969,0.177864,0.176373,0.096201,True,False,False,False,True,False


Provisional train-side policy: mixture_p035 — smallest peripheral-selection probability meeting the declared budgets; not a GPU authorization.


## I. Future HDF5 compatibility and limitations

**Do not pass these replacement schedules directly to the current HDF5BatchIterableDataset.** It sorts all supplied indices globally and forms fixed blocks. That would discard sampled ordering, change batch composition and make drop_last omit high-ID draws rather than a random suffix. Duplicate rows can also violate h5py's strictly increasing fancy-index requirement. Its current interface expects original split membership, not a replacement schedule.

A future separately authorized implementation needs to preserve schedule batch boundaries, order and multiplicity; for I/O it may read unique sorted IDs **within each batch** and restore duplicates/order with an inverse map. Do not globally deduplicate schedules. Partition batches once across workers; worker-specific seeds must not silently duplicate full schedules or alter intended probabilities. Keep 400000 drawn rows, 399872 consumed examples, 1562 optimizer steps, and epoch-specific deterministic streams. Random widely separated rows will probably reduce HDF5 locality; performance and VM memory/throughput remain unmeasured. No benchmark or loader change is made here.

Repeated examples reduce diversity, alter gradient noise and may overemphasize waveform/noise realizations; independent draws are not new independent data. Weight ESS measures probability concentration only; realized count ESS and distinct rows measure different exposure properties. Unique row IDs do not establish unique astrophysical sources. Joint label correlations mean balanced selection of one label's pool need not achieve equal marginal exposure of all labels, and a scalar sampling law cannot reproduce per-label weighted losses.

Sampling changes effective risk, even with ordinary MSE. Do not multiply these draws by M12.1/M12.2 loss weights or call the result original-distribution ERM. All evaluations and any future checkpoint selection must retain the established unweighted validation objective, while calibration/test/real events remain outside policy selection. This diagnostic does not establish model behavior, cross-seed reproducibility or scientific improvement. M11's partial-batch omission and other historical results are documented, not repaired.

## J. Minimal manual execution
Use the existing local Python 3.10/project environment and normal CBC_PE_DATA_ROOT convention. Run B–D sequentially, stopping on missing artifacts, empty pools or hash mismatches. Then run E, explicitly run the in-memory single-epoch simulations in F, followed by analytic G and decision H. Do not Run All. No packages are installed and no signals, nontrain targets or prediction archives are needed.

Only NumPy/Pandas/HDF5 operations and existing reference helpers are used; importing the helper's PyTorch dependency does not invoke GPU operations. Memory remains O(N) with small three-label arrays and one schedule at a time, never O(epochs×N). No schedules, weights, figures or reports are saved. The present deliverable contains no executed checks, simulated outputs or policy selection result. Runtime reference agreement and the actual coverage tradeoff remain pending manual execution.


## K. Scientific conclusions and decision — M12.3 sampling diagnostics

### K.1. Objective and scope

This diagnostic investigated the statistical and operational consequences of introducing marginal-mixture balanced sampling for the frozen M11 G1 training dataset.

The objective was to increase training exposure to marginally peripheral chirp-mass, total-mass, and effective-spin examples while controlling probability concentration, repeated observations, and loss of sample diversity.

All sampling probabilities and reference distributions were derived exclusively from the 400,000 training labels. No model training, inference, validation-based policy selection, calibration, test evaluation, or real-event analysis was performed.

The proposed sampler retains ordinary minibatch MSE but modifies the effective empirical training objective through nonuniform observation-selection probabilities.

### K.2. Compared sampling policies

Each training example was classified as marginally peripheral or central for each regression target using train-only standardized absolute-distance percentiles, with the peripheral boundary defined by q ≥ 0.8.

Three marginal-mixture sampling policies were analyzed:

- **p = 0.20:** near-uniform-with-replacement sanity control.
- **S1, p = 0.35:** moderate peripheral oversampling.
- **S2, p = 0.50:** stronger peripheral oversampling.

The original M11 without-replacement training exposure and uniform sampling with replacement were retained as distinct reference mechanisms.

Both candidates preserved strictly positive, normalized sampling probabilities, valid train-row alignment, deterministic schedule generation, and adequate representation of the examined discordant mass–spin regions.

### K.3. Induced training exposure

Under S1, the expected fraction of draws belonging to marginal P80 regions increased from approximately 20% to:

| Region | Original training distribution | S1 (p = 0.35) | S2 (p = 0.50) |
|---|---:|---:|---:|
| Chirp-mass P80 | 20.00% | 28.89% | 37.79% |
| Total-mass P80 | 20.00% | 28.82% | 37.64% |
| Effective-spin P80 | 20.00% | 24.81% | 29.62% |
| Global Mahalanobis P80 | 20.00% | 24.91% | 29.82% |

The induced marginal exposure is not symmetric despite selecting among the three parameters with equal probability. The strong overlap between chirp-mass and total-mass peripheral populations results in greater amplification of mass peripherality than effective-spin peripherality.

Therefore, the sampler does not produce equal marginal distributions. It preferentially increases the exposure of certain peripheral populations while preserving the observed joint label structure.

### K.4. Probability concentration and effective sample size

For S1, the effective scalar weights N_train × probability ranged approximately from 0.8125 to 1.7500, with a weight-based ESS of 373,700 observations, corresponding to 93.42% of the training population.

S2 produced more concentrated effective weights, ranging approximately from 0.6250 to 2.5000, with ESS/N = 78.03%.

These weights are induced by sampling probabilities and apply to the entire multivariate sample loss. They are not equivalent to M12.2's independent per-label loss weights, nor are they identical to M12.1's continuous Mahalanobis weighting.

Although the minibatch loss remains ordinary MSE, the expected training objective is probability-weighted empirical risk rather than the original uniformly weighted M11 risk.

### K.5. Sample diversity and replacement cost

Using the historical effective M11 exposure of 399,872 consumed observations and 1,562 optimizer steps per epoch:

| Sampling mechanism | Expected unique train rows per epoch | Fraction of train |
|---|---:|---:|
| M11 without replacement | 399,872 | 99.97% |
| Uniform with replacement | 252,801 | 63.20% |
| S1 (p = 0.35) | 248,068 | 62.02% |
| S2 (p = 0.50) | 235,082 | 58.77% |

The dominant per-epoch diversity reduction originates from sampling with replacement, rather than from the additional marginal balancing.

S1 produces approximately 151,932 omitted training rows and 151,804 duplicate draws per consumed epoch.

The expected unique-row coverage of S1 is 98.13% of uniform-with-replacement coverage at the same draw count, but substantially below the original M11 without-replacement exposure.

Under independently regenerated epoch schedules, expected cumulative coverage for S1 reaches approximately 84.83% after two epochs, 98.86% after five epochs, and 99.98% after ten epochs.

These calculations describe visitation probability, not effective information content or model generalization. Repeated observations do not represent additional independent astrophysical signals or noise realizations.

### K.6. Discordant mass–spin regions

S1 changes the expected draw frequency of the discordant regions as follows:

| Region | Original | S1 |
|---|---:|---:|
| Chirp mass peripheral / spin central | 16.18% | 22.48% |
| Chirp mass central / spin peripheral | 16.18% | 18.39% |
| Total mass peripheral / spin central | 16.42% | 22.74% |
| Total mass central / spin peripheral | 16.42% | 18.74% |

The sampler preferentially increases the exposure of mass-peripheral / spin-central examples relative to the reverse discordant configurations.

Consequently, S1 should not be interpreted as a targeted intervention for the mass-central / spin-peripheral deterioration identified during M12.1 and M12.2 evaluation.

Its primary mechanistic hypothesis is instead that increased exposure to peripheral populations, especially in mass parameters, may improve tail regression while maintaining acceptable global predictive performance.

### K.7. Policy-selection decision

The predefined train-side diagnostic budgets required:

1. Increased expected exposure in all three marginal P80 regions.
2. Weight-based ESS/N ≥ 0.90.
3. Expected unique-row coverage ≥ 95% of uniform-with-replacement coverage.
4. Maximum effective scalar weight ≤ 2.0.
5. Adequate original train support in the investigated discordant regions.

**S1 (p = 0.35) satisfied all five criteria. S2 (p = 0.50) failed the ESS, coverage, and maximum-weight criteria.**

**Decision: provisionally select S1 (p = 0.35) as the first M12.3 balanced-sampling training policy.**

This is a train-side experimental-design decision, not evidence of improved prediction accuracy.

The diagnostic thresholds are pragmatic design constraints rather than statistically validated optimality criteria.

### K.8. Implementation requirements and limitations

The existing HDF5 batch loader cannot consume replacement schedules directly without potentially changing batch order, multiplicities, and the intended sampling distribution.

A future implementation must preserve batch boundaries, sampled row order, repeated observations, deterministic epoch-level schedules, worker partitioning, and historical optimizer-step counts.

Efficient within-batch sorted-unique HDF5 reads followed by reconstruction of the sampled order should be investigated without changing the mathematical sampling law.

Random access may reduce I/O throughput and increase GPU idle time. This was not benchmarked by the current train-only diagnostic.

M12.3 also changes two properties simultaneously relative to M11: sampling probabilities and replacement behavior. Therefore, a difference in validation performance cannot be attributed uniquely to peripheral balancing without an additional matched sampling control or a different controlled design.

The weight-based ESS does not represent the number of distinct examples used during an epoch. Sample IDs also do not independently establish astrophysical-source independence.

No evidence about prediction accuracy, regression slopes, tail biases, calibration, domain transfer, or training-seed reproducibility can be inferred from these sampling diagnostics alone.

### K.9. Next experimental stage

Proceed to a separately controlled M12.3-S1 implementation, preserving:

- The frozen G1 dataset, source identifiers, and train/validation/calibration/test splits.
- M11 architecture, preprocessing, input normalization, label scaling, initialization seed, optimizer, and training configuration.
- Ordinary unweighted training MSE and unweighted validation-based checkpoint selection.
- The original number of optimizer steps per epoch.
- Separate checkpoint and artifact naming.
- Strict exclusion of calibration, test, and real-event data from policy selection.

After training, compare M12.3-S1 with M11, M12.1, and M12.2 on the identical validation examples using the established global, tail, SNR, Mahalanobis, and paired-bootstrap metrics.

Particular attention should be paid to whether mass-tail improvements are achieved without worsening central mass regression, effective-spin accuracy, or low-SNR performance.

Uniform-with-replacement sampling remains a potential future control for separating the balancing effect from the replacement effect, but no additional GPU training is authorized at this stage.

**Status: M12.3 train-only sampling diagnostics completed. S1 (p = 0.35) selected provisionally for implementation; predictive benefit remains untested.**